# Preparing a Xenium dataset for analysis

This notebook builds the workshop data **from 10x Genomics' raw files**. You can rerun it after the workshop to reproduce the data exactly, prepare the **whole slide** instead of the workshop crop, or adapt it to **another Xenium dataset**.

It produces, from one run:

| Output | Used for |
|---|---|
| `<NAME>.zarr`: a [SpatialData](https://spatialdata.scverse.org) object, ready for [Sopa](https://gustaveroussy.github.io/sopa/) | Day 2 (Python) |
| `qupath/`: OME-TIFF images and GeoJSON annotations | Day 1 (QuPath) |

Both describe exactly the same region, in the same coordinates: the top-left corner of the region is (0, 0), in pixels and in µm.

**Data:** 10x Genomics, *Xenium In Situ Gene and Protein Expression data for FFPE Human Renal Cell Carcinoma* ([dataset page](https://www.10xgenomics.com/datasets/xenium-protein-ffpe-human-renal-carcinoma)), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**What you need:**

- The workshop environment (`pixi install`). Start JupyterLab with `pixi run lab`.
- About **80 GB** of free disk for the raw download (36 GB, plus the same again unzipped), and a good internet connection. The whole slide is best prepared on a workstation or HPC.

**Running it without opening Jupyter** (for example on an HPC), from the repository folder:

```bash
XENIUM_RAW_DIR=/path/to/raw XENIUM_OUTPUT_DIR=/path/to/output \
  pixi run jupyter nbconvert --to notebook --execute notebooks/extra/prepare_xenium_dataset.ipynb --output-dir /path/to/output
```

Relative folders count from the repository folder. Set `XENIUM_DOWNLOAD=0` to reuse files already downloaded, `XENIUM_CROP=full` to prepare the whole slide, or `XENIUM_CROP=x_min,y_min,x_max,y_max` (µm) for a different region.

## 1. Settings

Everything specific to this dataset is in this cell. To use another Xenium dataset, change the file names and the crop.

In [ ]:
import json
import os
import re
import shutil
import time
import warnings
import zipfile
from pathlib import Path

import dask.array as da
import geopandas as gpd
import numpy as np
import pandas as pd
import requests
import spatialdata as sd
import spatialdata_io
import tifffile
import zarr
from shapely import affinity
from shapely.geometry import MultiPolygon, Polygon, box, mapping, shape
from spatialdata.models import Image2DModel, PointsModel, ShapesModel, TableModel
from spatialdata.transformations import Affine, Identity, Scale

warnings.filterwarnings("ignore", category=UserWarning)  # spatialdata/scanpy chatter; errors still show

# The repository folder (the one containing pixi.toml), so relative paths work wherever Jupyter starts
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pixi.toml").exists()), Path.cwd())

# Where 10x publishes the files: each URL is DOWNLOAD_PREFIX + a file name below
DOWNLOAD_PREFIX = (
    "https://cf.10xgenomics.com/samples/xenium/4.0.0/Xenium_V1_Human_Kidney_FFPE_Protein_updated/"
    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_"
)
XENIUM_ZIP = "outs.zip"                  # the full Xenium output bundle
HE_IMAGE = "he_image.ome.tif"            # post-Xenium H&E; set to None if the dataset has none
HE_ALIGNMENT = "he_imagealignment.csv"   # H&E -> Xenium alignment; None if no H&E
ANNOTATIONS = "annotation.geojson"       # pathology annotations in H&E pixels; None if there are none
SOURCE_URL = "https://www.10xgenomics.com/datasets/xenium-protein-ffpe-human-renal-carcinoma"

def folder_setting(variable, default):
    """A folder from an environment variable or the default. Relative paths count from the repository
    folder, because Jupyter runs notebooks from their own folder (notebooks/extra/)."""
    path = Path(os.environ.get(variable, default))
    return path if path.is_absolute() else ROOT / path


NAME = os.environ.get("XENIUM_NAME", "xenium_rcc_crop")
RAW_DIR = folder_setting("XENIUM_RAW_DIR", "data/raw")
OUTPUT_DIR = folder_setting("XENIUM_OUTPUT_DIR", "data/workshop")
DOWNLOAD = os.environ.get("XENIUM_DOWNLOAD", "1") == "1"  # set to "0" to use files already in RAW_DIR

# Region to keep, in µm on the full slide: (x_min, y_min, x_max, y_max). None keeps the whole slide.
CROP_UM = (1487.5, 0.0, 2987.5, 1500.0)  # the workshop crop: 1.5 x 1.5 mm, tumour + immune infiltration
if "XENIUM_CROP" in os.environ:
    value = os.environ["XENIUM_CROP"]
    CROP_UM = None if value == "full" else tuple(float(v) for v in value.split(","))

PROTEIN_IMAGE_OFFSET = 100   # Xenium Onboard Analysis 4+ adds this to every protein image pixel
WRITE_ZARR = True            # Day 2: the SpatialData Zarr
WRITE_QUPATH = True          # Day 1: OME-TIFFs and GeoJSON for QuPath
VALIDATE_WITH_SOPA = os.environ.get("XENIUM_VALIDATE", "1") == "1"  # segment a small tile as a final check
MAKE_ZIP = os.environ.get("XENIUM_ZIP", "0") == "1"  # zip the outputs for upload (e.g. to Zenodo)

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw files: {RAW_DIR}\nOutputs:   {OUTPUT_DIR}\nCrop (µm): {CROP_UM or 'whole slide'}")

## 2. Download and unzip

Downloads resume if interrupted, and files that are already complete are skipped. This step needs a good connection: `outs.zip` alone is 36 GB.

In [ ]:
HEADERS = {"User-Agent": "spatial-bio-training (workshop data preparation)"}
GB = 1024**3


def remote_size(url):
    response = requests.head(url, headers=HEADERS, allow_redirects=True, timeout=60)
    response.raise_for_status()
    return int(response.headers["Content-Length"])


def download(url, dest, attempts=10):
    """Download url to dest, resuming a partial file with explicit byte ranges."""
    total = remote_size(url)
    for attempt in range(attempts):
        have = dest.stat().st_size if dest.exists() else 0
        if have == total:
            print(f"{dest.name}: complete ({total / GB:.2f} GB)")
            return
        if have > total:
            dest.unlink()
            have = 0
        print(f"{dest.name}: downloading from {have / GB:.2f} of {total / GB:.2f} GB")
        try:
            # An explicit start-end range: the 10x CDN ignores open-ended suffix ranges
            headers = {**HEADERS, "Range": f"bytes={have}-{total - 1}"}
            with requests.get(url, headers=headers, stream=True, timeout=120) as response:
                response.raise_for_status()
                if have and response.status_code != 206:
                    have = 0  # the server ignored the range: start again
                with open(dest, "ab" if have else "wb") as handle:
                    for chunk in response.iter_content(8 * 1024 * 1024):
                        handle.write(chunk)
        except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as error:
            print(f"  interrupted ({type(error).__name__}), retrying in 10 s")
            time.sleep(10)
    raise RuntimeError(f"Could not download {url}")


optional_files = [f for f in (HE_IMAGE, HE_ALIGNMENT, ANNOTATIONS) if f]
if DOWNLOAD:
    for name in [XENIUM_ZIP, *optional_files]:
        download(DOWNLOAD_PREFIX + name, RAW_DIR / name)

# Unzip once; a marker file records that it finished
outs_root = RAW_DIR / "outs"
if not (outs_root / ".extracted").exists():
    print(f"Unzipping {XENIUM_ZIP} (this takes a while)...")
    with zipfile.ZipFile(RAW_DIR / XENIUM_ZIP) as archive:
        archive.extractall(outs_root)
    (outs_root / ".extracted").touch()

# The Xenium output folder is the one holding experiment.xenium
OUTS = min(outs_root.rglob("experiment.xenium"), key=lambda p: len(p.parts)).parent
print("Xenium output folder:", OUTS)

## 3. Read the Xenium output

`spatialdata_io.xenium` reads everything into one SpatialData object: the morphology images, transcripts, cell and nucleus boundaries, and the cell-by-feature table. Nothing large is loaded yet: images and transcripts are read lazily, only when needed.

- `gex_only=False` keeps the **protein** measurements in the table. The default drops them.
- Cell and nucleus *label images* are skipped: the boundary polygons carry the same information.

In [ ]:
experiment = json.loads((OUTS / "experiment.xenium").read_text())
PIXEL_SIZE = float(experiment["pixel_size"])  # µm per pixel of the morphology images
version = experiment.get("analysis_sw_version", "")
XOA_MAJOR = int(re.search(r"(\d+)\.", version).group(1)) if re.search(r"(\d+)\.", version) else 0
print(f"{experiment.get('region_name')}: XOA {version}, {PIXEL_SIZE} µm/px, {experiment.get('num_cells'):,} cells")

start = time.time()
xenium = spatialdata_io.xenium(
    OUTS,
    gex_only=False,
    cells_labels=False,
    nucleus_labels=False,
    morphology_mip=False,
    aligned_images=False,
)
print(f"Read in {time.time() - start:.0f} s")
xenium

## 4. Choose the region

The crop is given in µm but snapped to whole image pixels, so the images and the vector data (transcripts, boundaries) share exactly the same origin. Everything is then shifted so the region's top-left corner is (0, 0).

In [ ]:
focus = xenium.images["morphology_focus"]["scale0"].image  # (c, y, x), in pixels
height, width = focus.sizes["y"], focus.sizes["x"]

if CROP_UM is None:
    CROP_UM = (0.0, 0.0, width * PIXEL_SIZE, height * PIXEL_SIZE)
c0, r0 = (max(0, round(v / PIXEL_SIZE)) for v in CROP_UM[:2])
c1, r1 = min(width, round(CROP_UM[2] / PIXEL_SIZE)), min(height, round(CROP_UM[3] / PIXEL_SIZE))
X0_UM, Y0_UM, X1_UM, Y1_UM = c0 * PIXEL_SIZE, r0 * PIXEL_SIZE, c1 * PIXEL_SIZE, r1 * PIXEL_SIZE
H, W = r1 - r0, c1 - c0

# Transforms shared by every element: "global" is in image pixels (as spatialdata-io uses), "microns" in µm
px_to_um = Scale([PIXEL_SIZE, PIXEL_SIZE], axes=("x", "y"))
um_to_px = Scale([1 / PIXEL_SIZE, 1 / PIXEL_SIZE], axes=("x", "y"))
print(f"Region: x {X0_UM:.1f}-{X1_UM:.1f} µm, y {Y0_UM:.1f}-{Y1_UM:.1f} µm = {W} x {H} px "
      f"({W * PIXEL_SIZE / 1000:.2f} x {H * PIXEL_SIZE / 1000:.2f} mm)")


def scale_factors(h, w, smallest=512):
    """Downsampling steps for an image pyramid, stopping once the smallest level is about `smallest` px."""
    factors = []
    while min(h, w) // 2 >= smallest:
        factors.append(2)
        h, w = h // 2, w // 2
    return factors

## 5. Protein images: remove the +100 offset

Xenium Onboard Analysis 4 and later adds 100 to every pixel of the **protein** images, so that background-subtracted values never go negative. We subtract it again (clipping at 0), so a pixel value of 0 means no signal. The other images (DAPI, 18S, the segmentation stains and the background channels) have no offset and are left alone.

Protein channels are recognised by name: they are the image channels that match a protein in the cell table.

In [ ]:
table_10x = xenium.tables["table"]
feature_types = table_10x.var["feature_types"].to_numpy()
protein_names = set(table_10x.var_names[feature_types == "Protein Expression"])

channels = [str(c) for c in focus.coords["c"].values]
is_protein = np.array([c in protein_names for c in channels])
print(f"{is_protein.sum()} of {len(channels)} image channels are proteins")

image = focus.data[:, r0:r1, c0:c1]
if XOA_MAJOR >= 4 and is_protein.any():
    corrected = da.clip(image.astype(np.int32) - PROTEIN_IMAGE_OFFSET, 0, None).astype(image.dtype)
    image = da.where(is_protein[:, None, None], corrected, image)
    print(f"Subtracted {PROTEIN_IMAGE_OFFSET} from the protein channels")

morphology = Image2DModel.parse(
    image,
    dims=("c", "y", "x"),
    c_coords=channels,
    scale_factors=scale_factors(H, W),
    chunks={"c": 1, "y": 2048, "x": 2048},
    transformations={"global": Identity(), "microns": px_to_um},
)

## 6. Transcripts, cells and nuclei

Transcripts and boundaries are stored in µm. We keep:

- transcripts inside the region;
- **whole cells whose centre is inside the region** (so cells on the edge stay whole), and their nuclei.

In [ ]:
def drop_old_transforms(frame):
    """Cropped copies keep the full slide's transforms in .attrs; clear them so we can attach the region's."""
    frame.attrs.clear()
    return frame


points = xenium.points["transcripts"]
inside = (points["x"] >= X0_UM) & (points["x"] < X1_UM) & (points["y"] >= Y0_UM) & (points["y"] < Y1_UM)
transcripts_df = drop_old_transforms(points[inside].compute().reset_index(drop=True))
transcripts_df["x"] -= X0_UM
transcripts_df["y"] -= Y0_UM
transcripts = PointsModel.parse(
    transcripts_df,
    coordinates={"x": "x", "y": "y", "z": "z"},
    feature_key="feature_name",
    instance_key="cell_id",
    transformations={"global": um_to_px, "microns": Identity()},
)

cells = xenium.shapes["cell_boundaries"]
centres = cells.geometry.centroid
keep = ((centres.x >= X0_UM) & (centres.x < X1_UM) & (centres.y >= Y0_UM) & (centres.y < Y1_UM)).to_numpy()
cells = drop_old_transforms(cells[keep].copy())
cells["geometry"] = cells.geometry.translate(-X0_UM, -Y0_UM)
cell_boundaries = ShapesModel.parse(cells, transformations={"global": um_to_px, "microns": Identity()})

nuclei = xenium.shapes["nucleus_boundaries"]
nuclei = drop_old_transforms(nuclei[nuclei["cell_id"].isin(cells.index)].copy())
nuclei["geometry"] = nuclei.geometry.translate(-X0_UM, -Y0_UM)
nucleus_boundaries = ShapesModel.parse(nuclei, transformations={"global": um_to_px, "microns": Identity()})

print(f"{len(transcripts_df):,} transcripts, {len(cells):,} cells, {len(nuclei):,} nuclei")

## 7. Two tables: genes and proteins

10x's table mixes very different measurements: transcript **counts** for genes, mean stain **intensities** for proteins, and control features for quality checks. Counts and intensities need different normalisation, and nine names (CD4, CD8A, ...) are both a gene and a protein. So we split it into two tables over the same cells:

- `table`: gene counts. The control features are dropped, but each cell's control counts are kept as columns in `.obs`.
- `protein`: protein intensities (already offset-free in 10x's table).

Both are linked to `cell_boundaries` through `cell_id`.

In [ ]:
in_region = table_10x.obs["cell_id"].isin(cells.index).to_numpy()


def cell_table(feature_type):
    """The cells in the region, with features of one type, linked to cell_boundaries."""
    subset = table_10x[in_region, feature_types == feature_type].copy()
    subset.uns.pop("spatialdata_attrs", None)
    subset.obs["region"] = pd.Categorical(["cell_boundaries"] * subset.n_obs)
    if "spatial" in subset.obsm:  # cell centres, in µm
        subset.obsm["spatial"] = np.asarray(subset.obsm["spatial"]) - [X0_UM, Y0_UM]
    return TableModel.parse(subset, region="cell_boundaries", region_key="region", instance_key="cell_id")


tables = {"table": cell_table("Gene Expression")}
if (feature_types == "Protein Expression").any():
    tables["protein"] = cell_table("Protein Expression")
for key, value in tables.items():
    print(f"{key}: {value.n_obs:,} cells x {value.n_vars} features")

## 8. The H&E image and the pathology annotations

The H&E was scanned separately after the Xenium run, so it is rotated and at a different pixel size. 10x provides an **alignment matrix** that maps H&E pixels onto Xenium pixels.

- In the **Zarr**, we keep the H&E as scanned (cropped) and attach the alignment as a SpatialData *transformation*, so it lines up with everything else when plotted.
- For **QuPath**, which has no such transformations, we *resample* the H&E into the Xenium image's frame and pixel size (section 10).

The pathology annotations were drawn on the H&E, so the same matrix moves them onto the Xenium image.

In [ ]:
he_image, he_matrix, annotations = None, None, None

if HE_IMAGE and HE_ALIGNMENT and (RAW_DIR / HE_IMAGE).exists():
    alignment = np.loadtxt(RAW_DIR / HE_ALIGNMENT, delimiter=",")  # (x, y, 1) H&E px -> Xenium px

    # Read the full-resolution H&E lazily, tile by tile, instead of loading 3-4 GB at once
    he_full = da.from_zarr(zarr.open(tifffile.imread(RAW_DIR / HE_IMAGE, aszarr=True, level=0), mode="r"))  # (y, x, 3)

    # The part of the H&E that covers the region, plus a small margin
    corners = np.array([[c0, r0, 1], [c1, r0, 1], [c0, r1, 1], [c1, r1, 1]], dtype=float).T
    he_x, he_y = (np.linalg.inv(alignment) @ corners)[:2]
    hx0, hy0 = max(0, int(np.floor(he_x.min())) - 2), max(0, int(np.floor(he_y.min())) - 2)
    hx1 = min(he_full.shape[1], int(np.ceil(he_x.max())) + 2)
    hy1 = min(he_full.shape[0], int(np.ceil(he_y.max())) + 2)
    he_crop = he_full[hy0:hy1, hx0:hx1, :]

    # H&E-crop pixels -> region pixels: shift into the full H&E, align, then shift to the region's origin
    def translation(tx, ty):
        return np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], dtype=float)

    he_matrix = translation(-c0, -r0) @ alignment @ translation(hx0, hy0)
    um_matrix = np.diag([PIXEL_SIZE, PIXEL_SIZE, 1.0]) @ he_matrix
    he_image = Image2DModel.parse(
        he_crop.transpose(2, 0, 1),
        dims=("c", "y", "x"),
        c_coords=["r", "g", "b"],
        scale_factors=scale_factors(*he_crop.shape[:2]),
        chunks={"c": 3, "y": 2048, "x": 2048},
        transformations={
            "global": Affine(he_matrix, input_axes=("x", "y"), output_axes=("x", "y")),
            "microns": Affine(um_matrix, input_axes=("x", "y"), output_axes=("x", "y")),
        },
    )
    print(f"H&E: {he_crop.shape[1]} x {he_crop.shape[0]} px covering the region")

    if ANNOTATIONS and (RAW_DIR / ANNOTATIONS).exists():
        features = json.loads((RAW_DIR / ANNOTATIONS).read_text())
        features = features["features"] if isinstance(features, dict) else features
        # Annotations are in full-H&E pixels: align them, then shift to the region's origin
        (a, b, tx), (d, e, ty) = alignment[0], alignment[1]
        region_box = box(0, 0, W, H)
        rows, colours = [], {}
        for feature in features:
            props = feature.get("properties", {})
            classification = props.get("classification") or {}
            # 10x's file is inconsistent: some annotations have only a name. Use the name as the class.
            name = props.get("name") or classification.get("name") or "Unclassified"
            colour = colours.setdefault(name, classification.get("color") or props.get("color") or [128, 128, 128])
            geometry = affinity.affine_transform(shape(feature["geometry"]), [a, b, d, e, tx - c0, ty - r0])
            geometry = geometry.intersection(region_box)
            parts = [geometry] if isinstance(geometry, Polygon) else [
                g for g in getattr(geometry, "geoms", []) if isinstance(g, Polygon)]
            parts = [p for p in parts if p.area > 0]
            if parts:
                rows.append({"name": name, "colour": "#{:02x}{:02x}{:02x}".format(*colour),
                             "geometry": parts[0] if len(parts) == 1 else MultiPolygon(parts)})
        if rows:
            annotations = ShapesModel.parse(
                gpd.GeoDataFrame(rows, geometry="geometry"),
                transformations={"global": Identity(), "microns": px_to_um},
            )
            print("Annotations in the region:", annotations["name"].value_counts().to_dict())
        else:
            print("No annotations fall inside the region")
else:
    print("No H&E image: skipping the H&E and annotations")

## 9. Assemble and save the SpatialData Zarr

The Zarr records where the region came from (`attrs`), so it can always be traced back to 10x's files.

> **Using Sopa on this data:** `sopa.aggregate` writes to a table called `table` by default, which would replace 10x's gene table. Pass your own name, e.g. `key_added="instanseg_table"`. Also pass `image_key="morphology_focus"`, since there are two images.

In [ ]:
images = {"morphology_focus": morphology}
shapes = {"cell_boundaries": cell_boundaries, "nucleus_boundaries": nucleus_boundaries}
if he_image is not None:
    images["he"] = he_image
if annotations is not None:
    shapes["pathology_annotations"] = annotations

sdata = sd.SpatialData(
    images=images,
    points={"transcripts": transcripts},
    shapes=shapes,
    tables=tables,
    attrs={
        "source": SOURCE_URL,
        "licence": "CC BY 4.0, 10x Genomics",
        "xenium_analysis_version": version,
        "pixel_size_um": PIXEL_SIZE,
        "region_on_slide_um": [X0_UM, Y0_UM, X1_UM, Y1_UM],
        "region_on_slide_px": [c0, r0, c1, r1],
        "protein_image_offset_removed": PROTEIN_IMAGE_OFFSET if XOA_MAJOR >= 4 and is_protein.any() else 0,
        "prepared_by": "spatial-bio-training notebooks/extra/prepare_xenium_dataset.ipynb",
    },
)

zarr_path = OUTPUT_DIR / f"{NAME}.zarr"
if WRITE_ZARR:
    if zarr_path.exists():
        shutil.rmtree(zarr_path)
    start = time.time()
    sdata.write(zarr_path)
    print(f"Wrote {zarr_path} in {time.time() - start:.0f} s")
sdata

## 10. Files for QuPath (Day 1)

QuPath can't open a SpatialData Zarr, so the same region is also written as files QuPath reads directly:

- `<NAME>_if.ome.tif`: all morphology/protein channels, offset-corrected, as a pyramid.
- `<NAME>_he.ome.tif`: the H&E, **resampled into the same frame and pixel size** as the IF image, so the two overlay exactly.
- `<NAME>_annotations.geojson`: the pathology annotations; `<NAME>_10x_cells.geojson`: 10x's cell boundaries.

All are in pixels with the region's corner at (0, 0), like the Zarr's `global` coordinate system. Import the GeoJSON files with *File › Import objects*.

In [ ]:
TILE = 512


def tiles(level, tile=TILE):
    """Yield tile x tile blocks (padded at the edges) of a CYX array, or of a YXS RGB array."""
    rgb = level.ndim == 3 and level.shape[-1] == 3 and level.dtype == np.uint8
    for plane in [level] if rgb else (level[c] for c in range(level.shape[0])):
        for y in range(0, plane.shape[0], tile):
            row = np.asarray(plane[y:y + tile])  # read a strip at a time
            for x in range(0, plane.shape[1], tile):
                block = row[:, x:x + tile]
                if block.shape[:2] != (tile, tile):
                    pad = [(0, tile - block.shape[0]), (0, tile - block.shape[1])] + [(0, 0)] * (block.ndim - 2)
                    block = np.pad(block, pad)
                yield block


def write_ome_tiff(path, levels, channel_names=None):
    """Write a tiled, pyramidal OME-TIFF that QuPath opens directly, without loading it all into memory."""
    rgb = levels[0].ndim == 3 and levels[0].shape[-1] == 3 and levels[0].dtype == np.uint8
    n_planes = 1 if rgb else levels[0].shape[0]
    metadata = {
        "axes": "YXS" if rgb else "CYX",
        "PhysicalSizeX": PIXEL_SIZE, "PhysicalSizeXUnit": "µm",
        "PhysicalSizeY": PIXEL_SIZE, "PhysicalSizeYUnit": "µm",
        # Where the region sits on the full slide (for reference; QuPath still starts at 0, 0)
        "Plane": {"PositionX": [X0_UM] * n_planes, "PositionXUnit": ["µm"] * n_planes,
                  "PositionY": [Y0_UM] * n_planes, "PositionYUnit": ["µm"] * n_planes},
    }
    if channel_names:
        metadata["Channel"] = {"Name": list(channel_names)}
    with tifffile.TiffWriter(path, bigtiff=True, ome=True) as tif:
        for i, level in enumerate(levels):
            options = dict(tile=(TILE, TILE), compression="zlib", dtype=level.dtype, shape=level.shape,
                           photometric="rgb" if rgb else "minisblack")
            if i == 0:
                tif.write(tiles(level), subifds=len(levels) - 1, metadata=metadata, **options)
            else:
                tif.write(tiles(level), subfiletype=1, metadata=None, **options)


def write_geojson(path, geometries, names, classes=None, colours=None, object_type="annotation"):
    """GeoJSON QuPath can import: every classification has a name and an RGB colour."""
    features = []
    for i, (geometry, name) in enumerate(zip(geometries, names)):
        properties = {"objectType": object_type, "name": str(name)}
        if classes is not None:
            rgb = [int(colours[i][j:j + 2], 16) for j in (1, 3, 5)]
            properties["classification"] = {"name": classes[i], "color": rgb}
        features.append({"type": "Feature", "geometry": mapping(geometry), "properties": properties})
    path.write_text(json.dumps({"type": "FeatureCollection", "features": features}))


if WRITE_QUPATH:
    qupath_dir = OUTPUT_DIR / "qupath"
    qupath_dir.mkdir(exist_ok=True)

    start = time.time()
    levels = [morphology[f"scale{i}"].image.data for i in range(len(morphology.children))]
    write_ome_tiff(qupath_dir / f"{NAME}_if.ome.tif", levels, channels)
    print(f"IF image written in {time.time() - start:.0f} s")

    if he_image is not None:
        from dask_image.ndinterp import affine_transform

        # For each output (region) pixel, where to sample in the H&E crop. scipy works in (row, col) order.
        inverse = np.linalg.inv(he_matrix)
        swap = np.array([[0, 1], [1, 0]])
        matrix, offset = swap @ inverse[:2, :2] @ swap, swap @ inverse[:2, 2]
        he_aligned = da.stack(
            [affine_transform(he_crop[..., i].astype(np.float32), matrix, offset=offset, output_shape=(H, W),
                              output_chunks=(2048, 2048), order=1, cval=255) for i in range(3)],
            axis=-1,
        ).clip(0, 255).astype(np.uint8)
        he_levels = [he_aligned]
        for _ in scale_factors(H, W):
            he_levels.append(da.coarsen(np.mean, he_levels[-1], {0: 2, 1: 2}, trim_excess=True).astype(np.uint8))
        start = time.time()
        write_ome_tiff(qupath_dir / f"{NAME}_he.ome.tif", he_levels)
        print(f"Resampled H&E written in {time.time() - start:.0f} s")

    if annotations is not None:
        write_geojson(qupath_dir / f"{NAME}_annotations.geojson", annotations.geometry, annotations["name"],
                      classes=list(annotations["name"]), colours=list(annotations["colour"]))

    # 10x's cell boundaries, from µm to pixels
    cells_px = cells.geometry.scale(1 / PIXEL_SIZE, 1 / PIXEL_SIZE, origin=(0, 0))
    write_geojson(qupath_dir / f"{NAME}_10x_cells.geojson", cells_px, cells.index, object_type="detection")
    print("QuPath files:", sorted(p.name for p in qupath_dir.iterdir()))

## 11. Check the result

Read the Zarr back and, as a final check that it works with Sopa, segment one small tile with InstanSeg, the same model used in QuPath on Day 1.

In [ ]:
if WRITE_ZARR:
    check = sd.read_zarr(zarr_path)
    focus_check = check.images["morphology_focus"]["scale0"].image
    assert focus_check.sizes["x"] == W and focus_check.sizes["y"] == H
    assert check.tables["table"].n_obs == len(check.shapes["cell_boundaries"])
    print(check)

if WRITE_ZARR and VALIDATE_WITH_SOPA:
    import sopa

    # A tile from the middle of the region (an edge may be mostly empty background)
    size = min(1024, H, W)
    x0, y0 = (W - size) // 2, (H - size) // 2
    tile = sd.bounding_box_query(check, axes=("x", "y"), min_coordinate=[x0, y0],
                                 max_coordinate=[x0 + size, y0 + size], target_coordinate_system="global")
    tile = sd.SpatialData(images={"morphology_focus": tile.images["morphology_focus"]})
    sopa.make_image_patches(tile, patch_width=size, image_key="morphology_focus")
    sopa.segmentation.instanseg(tile, channels=["DAPI"], pixel_size=PIXEL_SIZE, image_key="morphology_focus")
    print(f"Sopa + InstanSeg found {len(tile.shapes['instanseg_boundaries'])} cells in a {size} x {size} px tile")

## 12. Notes for the archive (optional)

Writes a short README with the attribution, and, if `MAKE_ZIP` is set, zips the Zarr and QuPath files for upload (for example to Zenodo).

In [ ]:
readme = f"""# {NAME}

Prepared from 10x Genomics' "Xenium In Situ Gene and Protein Expression data for FFPE Human Renal Cell Carcinoma"
({SOURCE_URL}), licensed under CC BY 4.0. Please cite 10x Genomics when using these data.

Region: x {X0_UM:.1f}-{X1_UM:.1f} µm, y {Y0_UM:.1f}-{Y1_UM:.1f} µm of the full slide ({W} x {H} px at {PIXEL_SIZE} µm/px).
All files use region coordinates: the region's top-left corner is (0, 0).
Changes from the original: cropped; {PROTEIN_IMAGE_OFFSET if XOA_MAJOR >= 4 else 0} subtracted from protein images;
cell table split into gene (`table`) and protein (`protein`) tables; H&E resampled into the Xenium frame for QuPath.

Made with notebooks/extra/prepare_xenium_dataset.ipynb in https://github.com/FrancisCrickInstitute/spatial-bio-training
"""
(OUTPUT_DIR / "README.md").write_text(readme)

if MAKE_ZIP:
    for folder in [p for p in (zarr_path, OUTPUT_DIR / "qupath") if p.exists()]:
        archive = shutil.make_archive(str(folder), "zip", root_dir=folder.parent, base_dir=folder.name)
        print("Wrote", archive)